# Project Progress Report - From Early Attempts to Final Hough + HSV/LAB

This notebook presents the **project progression** in a clear, defense-ready format.

What this report shows:
- methods tried earlier,
- measurable behavior of each method,
- why we moved to Hough circles,
- why center-ring color analysis was needed for 1 EUR / 2 EUR robustness.

## Contents

1. Setup and dataset loading
2. Attempt A: Otsu thresholding + connected components
3. Attempt B: Canny + contour circularity filtering
4. Attempt C: Hough circles (turning point)
5. Comparison table and chart
6. Color-stage evolution (global HSV -> center-ring LAB/HSV)
7. Final pipeline benchmark and case study
8. Final conclusions

In [ ]:
from pathlib import Path
import sys
import inspect

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT / "Image_projet_money" / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT / "Image_projet_money"
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

try:
    import cv2
    CV2_AVAILABLE = True
except Exception as e:
    CV2_AVAILABLE = False
    print("OpenCV not available:", e)

from src.config import DetectionConfig, RuntimeConfig
from src.dataset import DatasetRepository
from src.io_utils import ImagePathResolver
from src.processor import CoinProcessor
from src.processor_color import CoinColorClassifier

df_gt = DatasetRepository().to_dataframe()
resolver = ImagePathResolver(RuntimeConfig().IMAGE_DIRECTORY)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Annotated images:", len(df_gt))
print("CV2_AVAILABLE:", CV2_AVAILABLE)

## 1) Dataset Snapshot

Before comparing methods, check the dataset distribution.

In [ ]:
display(df_gt.head(12))

group_summary = (
    df_gt.groupby("group")
    .agg(num_images=("image", "count"), avg_pieces=("pieces", "mean"), avg_value=("value_eur", "mean"))
    .reset_index()
    .sort_values("group")
)
display(group_summary)

## 2) Attempt A - Otsu Threshold + Connected Components

### Idea
Segment image into foreground/background, then count connected blobs.

### Why it was not enough
- merges touching/near coins,
- sensitive to shadows and uneven lighting,
- unstable on mixed backgrounds.

In [ ]:
def count_coins_threshold_otsu(img_bgr, min_area=250):
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    _, bw = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # Evaluate both polarity options and pick the one with more plausible connected components.
    counts = []
    for candidate in (bw, cv2.bitwise_not(bw)):
        n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(candidate)
        area = stats[:, cv2.CC_STAT_AREA]
        valid = (area >= min_area)
        # Exclude background label (0)
        comp_count = int(np.sum(valid[1:]))
        counts.append(comp_count)

    return max(counts)

## 3) Attempt B - Canny Edges + Contour Circularity

### Idea
Detect edges, extract contours, keep circular candidates.

### Improvement over attempt A
- stronger geometric prior,
- less dependent on absolute intensity.

### Remaining issues
- thin/fragmented edges under noise,
- frequent misses on low-contrast coin boundaries,
- fragile when multiple contours overlap.

In [ ]:
def count_coins_contour(img_bgr, min_radius=8, max_radius=200, min_circularity=0.65):
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    eq = cv2.equalizeHist(gray)
    edges = cv2.Canny(eq, 45, 130)

    contours_info = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    contours = contours_info[0] if len(contours_info) == 2 else contours_info[1]

    count = 0
    for contour in contours:
        area = cv2.contourArea(contour)
        perim = cv2.arcLength(contour, True)
        if perim <= 1e-6:
            continue

        circularity = (4.0 * np.pi * area) / (perim * perim)
        (x, y), r = cv2.minEnclosingCircle(contour)
        if r < min_radius or r > max_radius:
            continue
        if circularity < min_circularity:
            continue
        count += 1

    return count

## 4) Attempt C - Hough Circles (Turning Point)

### Idea
Coins are circles, so detect circles directly with Hough transform.

### Why this changed the project
- geometry-first detection became much more stable,
- better behavior under moderate illumination changes,
- cleaner path to later classification.

In [ ]:
def count_coins_hough(img_bgr, dp=1.2, minDist=70, p1=50, p2=45, minR=10, maxR=150):
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    gray = cv2.normalize(gray, None, alpha=0, beta=255, norm_type=cv2.NORM_MINMAX)
    blur = cv2.medianBlur(gray, 15)

    circles = cv2.HoughCircles(
        blur,
        cv2.HOUGH_GRADIENT,
        dp=dp,
        minDist=minDist,
        param1=p1,
        param2=p2,
        minRadius=minR,
        maxRadius=maxR,
    )
    if circles is None:
        return 0
    circles = np.squeeze(circles, axis=0)
    if circles.ndim == 1:
        circles = np.expand_dims(circles, axis=0)
    return int(circles.shape[0])

## 5) Evaluation Utility for Count-Only Methods

The same subset is evaluated with each method to compare count behavior.

In [ ]:
def evaluate_count_method(method_name, method_fn, max_images=25):
    rows = []
    eval_df = df_gt.head(max_images)

    for _, row in eval_df.iterrows():
        path = resolver.resolve(row["image"], row["group"])
        if not path:
            continue
        img = cv2.imread(path) if CV2_AVAILABLE else None
        if img is None:
            continue

        pred = int(method_fn(img))
        true = int(row["pieces"])
        rows.append(
            {
                "method": method_name,
                "image": row["image"],
                "group": row["group"],
                "pred_count": pred,
                "true_count": true,
                "abs_err": abs(pred - true),
            }
        )

    if not rows:
        return pd.DataFrame(), {}

    out = pd.DataFrame(rows)
    metrics = {
        "method": method_name,
        "images": len(out),
        "count_mae": float(out["abs_err"].mean()),
        "count_exact_rate_%": float((out["abs_err"] == 0).mean() * 100.0),
    }
    return out, metrics

if CV2_AVAILABLE:
    a_df, a_metrics = evaluate_count_method("A_otsu", count_coins_threshold_otsu, max_images=25)
    b_df, b_metrics = evaluate_count_method("B_contour", count_coins_contour, max_images=25)
    c_df, c_metrics = evaluate_count_method("C_hough", count_coins_hough, max_images=25)

    metrics_df = pd.DataFrame([a_metrics, b_metrics, c_metrics])
    display(metrics_df)
else:
    print("Skipping evaluation: OpenCV not available.")

## 6) Comparison Chart (Count MAE and Exact-Match Rate)

In [ ]:
if CV2_AVAILABLE and 'metrics_df' in globals() and not metrics_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))

    axes[0].bar(metrics_df["method"], metrics_df["count_mae"])
    axes[0].set_title("Count MAE (lower is better)")
    axes[0].set_ylabel("MAE")
    axes[0].grid(axis="y", alpha=0.25)

    axes[1].bar(metrics_df["method"], metrics_df["count_exact_rate_%"])
    axes[1].set_title("Exact Count Match Rate")
    axes[1].set_ylabel("Percent")
    axes[1].set_ylim(0, 100)
    axes[1].grid(axis="y", alpha=0.25)

    plt.tight_layout()
    plt.show()
else:
    print("No metrics to plot.")

## 7) Why Color Stage Was Still Needed After Hough

Hough solves mostly **where** coins are and **how many**.

Value prediction still needs material cues:
- bronze family -> 1c / 2c / 5c,
- gold family -> 10c / 20c / 50c,
- bimetal -> 1 EUR / 2 EUR.

An early color approach used global full-coin HSV statistics only, which was weaker for bimetal separation under warm lighting.

In [ ]:
def simple_hsv_material_score(img_bgr, circle):
    x, y, r = [int(v) for v in circle]
    h, w = img_bgr.shape[:2]
    yy, xx = np.ogrid[:h, :w]
    mask = (xx - x) ** 2 + (yy - y) ** 2 <= r ** 2

    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    pix = hsv[mask]
    if pix.size == 0:
        return {"sat": 0.0, "hue": 0.0}

    sat = float(np.mean(pix[:, 1])) / 255.0
    hue = float(np.mean(pix[:, 0])) * 2.0
    return {"sat": sat, "hue": hue}

if CV2_AVAILABLE:
    # Small demonstration on first detectable circle from one sample image
    row = df_gt.iloc[0]
    path = resolver.resolve(row["image"], row["group"])
    img = cv2.imread(path) if path else None
    if img is not None:
        cc = count_coins_hough(img)
        print("Detected circles (count-only):", cc)
        # This cell intentionally demonstrates simple score existence, not final classification quality.
else:
    print("Skipping simple HSV demonstration: OpenCV not available.")

## 8) Final Color Logic: Center-Ring Analysis (Current Code)

Current implementation uses center and ring region statistics in LAB/HSV and computes directional bimetal evidence.

The next cell prints real source snippets from the current `CoinColorClassifier`.

In [ ]:
def show_source(fn, max_lines=120):
    src = inspect.getsource(fn).splitlines()
    print("".join(src[:max_lines]))
    if len(src) > max_lines:
        print("... (truncated)")

print("[CoinColorClassifier.extract_coin_features]")
show_source(CoinColorClassifier.extract_coin_features, max_lines=130)

print("[CoinColorClassifier._compute_color_group_scores]")
show_source(CoinColorClassifier._compute_color_group_scores, max_lines=100)

## 9) Case Study: `gp5/0.jpeg`

Known reference in dataset:
- image: `0.jpeg`
- group: `gp5`
- ground truth: 2 coins, total value 2.20 EUR

This runs the **current final pipeline** on that case.

In [ ]:
if CV2_AVAILABLE:
    case_path = resolver.resolve("0.jpeg", "gp5")
    print("case_path:", case_path)
    if case_path:
        img = cv2.imread(case_path)
        proc = CoinProcessor(DetectionConfig())
        res = proc.execute(img, filename="0.jpeg")
        print("pred_count:", res.coin_count)
        print("pred_value_eur:", round(res.estimated_value_eur, 2))
        print("coin_labels:", res.coin_labels)
        print("color_labels:", res.coin_color_labels)

        # show final annotated stage
        final_step = res.steps[3] if len(res.steps) > 3 else res.steps[-1]
        plt.figure(figsize=(6, 5))
        plt.imshow(cv2.cvtColor(final_step.image, cv2.COLOR_BGR2RGB))
        plt.title("Final detection + labels")
        plt.axis("off")
        plt.show()
else:
    print("Skipping case study: OpenCV not available.")

## 10) Short Final-Pipeline Benchmark (Subset)

This evaluates count and value errors on a subset using the current full pipeline.

In [ ]:
def evaluate_final_pipeline(max_images=25):
    proc = CoinProcessor(DetectionConfig())
    rows = []

    for _, row in df_gt.head(max_images).iterrows():
        path = resolver.resolve(row["image"], row["group"])
        if not path:
            continue
        img = cv2.imread(path) if CV2_AVAILABLE else None
        if img is None:
            continue

        res = proc.execute(img, filename=row["image"])
        pred_count = int(res.coin_count)
        true_count = int(row["pieces"])

        pred_val = float(res.estimated_value_eur)
        true_val = float(row["value_eur"]) if pd.notna(row["value_eur"]) else np.nan

        rows.append(
            {
                "image": row["image"],
                "group": row["group"],
                "pred_count": pred_count,
                "true_count": true_count,
                "count_abs_err": abs(pred_count - true_count),
                "pred_value": pred_val,
                "true_value": true_val,
                "value_abs_err": abs(pred_val - true_val) if pd.notna(true_val) else np.nan,
            }
        )

    out = pd.DataFrame(rows)
    if out.empty:
        return out, {}

    metrics = {
        "images": len(out),
        "count_mae": float(out["count_abs_err"].mean()),
        "count_exact_rate_%": float((out["count_abs_err"] == 0).mean() * 100.0),
        "value_mae": float(out["value_abs_err"].dropna().mean()) if out["value_abs_err"].notna().any() else np.nan,
    }
    return out, metrics

if CV2_AVAILABLE:
    final_df, final_metrics = evaluate_final_pipeline(max_images=25)
    display(final_df.head(12))
    print("Final subset metrics:", final_metrics)
else:
    print("Skipping final benchmark: OpenCV not available.")

## 11) Ready-to-Present Summary

- Attempt A (thresholding) was fast but unstable under real image variability.
- Attempt B (contours) improved geometric reasoning but remained fragile in edge quality.
- Attempt C (Hough) was the key turning point for reliable circle detection.
- Final system combines Hough geometry with center-ring LAB/HSV color priors and a global scale fit.
- This combination made both count and denomination assignment substantially more defensible.